<a href="https://colab.research.google.com/github/gabrielcabanilla1/EXSU-500-Clinical-NLP-large-language-models-rct/blob/main/notebooks/02_Model_Gabriel.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!git clone --depth 1 https://github.com/Franck-Dernoncourt/pubmed-rct.git

Cloning into 'pubmed-rct'...
remote: Enumerating objects: 19, done.
remote: Counting objects: 100% (19/19), done.
remote: Compressing objects: 100% (14/14), done.
remote: Total 19 (delta 5), reused 14 (delta 5), pack-reused 0 (from 0)
Receiving objects: 100% (19/19), 177.08 MiB | 24.20 MiB/s, done.
Resolving deltas: 100% (5/5), done.
Updating files: 100% (13/13), done.


In [ ]:
CARPETA = 'pubmed-rct/PubMed_20k_RCT'

In [ ]:
import pandas as pd

def leer_archivo(ruta):
    filas = []
    oraciones = []
    abstract_id = None

    def guardar_abstract():
        n = len(oraciones)
        for i, (etiqueta, texto) in enumerate(oraciones):
            filas.append({
                'abstract_id': abstract_id,
                'etiqueta': etiqueta,
                'texto': texto,
                'posicion': i / max(n - 1, 1),
                'indice': i,
                'n_oraciones': n,
            })

    for linea in open(ruta, encoding='utf-8'):
        linea = linea.rstrip('\n')
        if linea.startswith('###'):
            abstract_id = linea[3:]
            oraciones = []
        elif linea.strip() == '':
            guardar_abstract()
            oraciones = []
        else:
            etiqueta, texto = linea.split('\t', 1)
            oraciones.append((etiqueta, texto))
    guardar_abstract()
    return pd.DataFrame(filas)

train = leer_archivo(f'{CARPETA}/train.txt')
dev   = leer_archivo(f'{CARPETA}/dev.txt')
test  = leer_archivo(f'{CARPETA}/test.txt')

train.head(12)

,abstract_id,etiqueta,texto,posicion,indice,n_oraciones
0,24293578,OBJECTIVE,To investigate the efficacy of 6 weeks of dail...,0.000000,0,12
1,24293578,METHODS,A total of 125 patients with primary knee OA w...,0.090909,1,12
2,24293578,METHODS,Outcome measures included pain reduction and i...,0.181818,2,12
3,24293578,METHODS,Pain was assessed using the visual analog pain...,0.272727,3,12
4,24293578,METHODS,Secondary outcome measures included the Wester...,0.363636,4,12
5,24293578,METHODS,"Serum levels of interleukin 1 ( IL-1 ) , IL-6 ...",0.454545,5,12
6,24293578,RESULTS,There was a clinically relevant reduction in t...,0.545455,6,12
7,24293578,RESULTS,The mean difference between treatment arms ( 9...,0.636364,7,12
8,24293578,RESULTS,"Further , there was a clinically relevant redu...",0.727273,8,12
9,24293578,RESULTS,These differences remained significant at 12 w...,0.818182,9,12


In [ ]:
for nombre, tabla in [('Entrenamiento', train), ('Validación', dev), ('Prueba', test)]:
    print(f"{nombre:14s} {tabla.abstract_id.nunique():8,d} abstracts  {len(tabla):10,d} oraciones")

Entrenamiento    15,000 abstracts     180,040 oraciones
Validación        2,500 abstracts      30,212 oraciones
Prueba            2,500 abstracts      30,135 oraciones


In [ ]:
CLASES = ['BACKGROUND', 'OBJECTIVE', 'METHODS', 'RESULTS', 'CONCLUSIONS']

for tabla in (train, dev, test):
    tabla['y'] = tabla.etiqueta.map({c: i for i, c in enumerate(CLASES)})

print(train.etiqueta.value_counts(normalize=True).round(3))

etiqueta
METHODS        0.330
RESULTS        0.322
CONCLUSIONS    0.151
BACKGROUND     0.121
OBJECTIVE      0.077
Name: proportion, dtype: float64


In [ ]:
import numpy as np
from sklearn.metrics import f1_score, accuracy_score

N_TRAMOS = 10

for tabla in (train, dev, test):
    tabla['tramo'] = np.minimum((tabla.posicion * N_TRAMOS).astype(int), N_TRAMOS - 1)

regla = train.groupby('tramo').etiqueta.agg(lambda s: s.mode()[0])
print('Regla aprendida:')
print(regla)

pred_posicion = dev.tramo.map(regla)

print('\nExactitud:', round(accuracy_score(dev.etiqueta, pred_posicion), 3))
print('Macro-F1: ', round(f1_score(dev.etiqueta, pred_posicion, average='macro'), 3))

Regla aprendida:
tramo
0     BACKGROUND
1        METHODS
2        METHODS
3        METHODS
4        METHODS
5        RESULTS
6        RESULTS
7        RESULTS
8        RESULTS
9    CONCLUSIONS
Name: etiqueta, dtype: object

Exactitud: 0.689
Macro-F1:  0.548


In [ ]:
import torch
dispositivo = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Dispositivo:', dispositivo)
if dispositivo == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))

Dispositivo: cuda
GPU: Tesla T4


In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')
CARPETA_DRIVE = '/content/drive/MyDrive/proyecto_exsu'
os.makedirs(CARPETA_DRIVE, exist_ok=True)

Mounted at /content/drive


In [ ]:
from transformers import AutoTokenizer

MODELO_BASE = 'microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext'
SUBMUESTRA = None
EPOCAS = 2
TAMANO_LOTE = 64
LARGO_MAXIMO = 128
TASA_APRENDIZAJE = 3e-5
SEMILLA = 42

tokenizador = AutoTokenizer.from_pretrained(MODELO_BASE)

def tokenizar(df):
    return tokenizador(list(df.texto), truncation=True, max_length=LARGO_MAXIMO,
                       padding='max_length', return_tensors='pt')

train_bert = train.sample(SUBMUESTRA, random_state=SEMILLA) if SUBMUESTRA else train
tok_train = tokenizar(train_bert)
tok_dev = tokenizar(dev)

print('Oraciones de entrenamiento:', len(train_bert))
print('Ejemplo:', tokenizador.convert_ids_to_tokens(tok_train['input_ids'][0][:15]))

config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/28.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/226k [00:00<?, ?B/s]

Oraciones de entrenamiento: 180040
Ejemplo: ['[CLS]', 'to', 'investigate', 'the', 'efficacy', 'of', '6', 'weeks', 'of', 'daily', 'low', '-', 'dose', 'oral', 'prednisolone']


In [ ]:
from transformers import AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from torch.utils.data import TensorDataset, DataLoader
import numpy as np

torch.manual_seed(SEMILLA)
np.random.seed(SEMILLA)

modelo_bert = AutoModelForSequenceClassification.from_pretrained(MODELO_BASE, num_labels=5).to(dispositivo)

datos_train = TensorDataset(tok_train['input_ids'], tok_train['attention_mask'],
                            torch.tensor(train_bert.y.values))
datos_dev = TensorDataset(tok_dev['input_ids'], tok_dev['attention_mask'])
cargador_train = DataLoader(datos_train, batch_size=TAMANO_LOTE, shuffle=True,
                            generator=torch.Generator().manual_seed(SEMILLA))

optimizador = torch.optim.AdamW(modelo_bert.parameters(), lr=TASA_APRENDIZAJE, weight_decay=0.01)
pasos_totales = EPOCAS * len(cargador_train)
programador = get_linear_schedule_with_warmup(optimizador, int(0.06 * pasos_totales), pasos_totales)
escalador = torch.amp.GradScaler(enabled=(dispositivo == 'cuda'))

@torch.no_grad()
def predecir_bert(datos):
    modelo_bert.eval()
    probs = []
    for ids, mascara in DataLoader(datos, batch_size=256):
        with torch.autocast(dispositivo, enabled=(dispositivo == 'cuda')):
            logits = modelo_bert(input_ids=ids.to(dispositivo), attention_mask=mascara.to(dispositivo)).logits
        probs.append(torch.softmax(logits.float(), dim=-1).cpu().numpy())
    return np.concatenate(probs)

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  440MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect ide

model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

In [ ]:
from tqdm.auto import tqdm
from sklearn.metrics import f1_score, accuracy_score
import time

ruta_bert = f'{CARPETA_DRIVE}/pubmedbert_texto'
mejor_f1, inicio = -1, time.time()

for epoca in range(EPOCAS):
    modelo_bert.train()
    for ids, mascara, y_lote in tqdm(cargador_train, desc=f'Época {epoca + 1}/{EPOCAS}'):
        optimizador.zero_grad()
        with torch.autocast(dispositivo, enabled=(dispositivo == 'cuda')):
            salida = modelo_bert(input_ids=ids.to(dispositivo), attention_mask=mascara.to(dispositivo),
                                 labels=y_lote.to(dispositivo))
        escalador.scale(salida.loss).backward()
        escalador.unscale_(optimizador)
        torch.nn.utils.clip_grad_norm_(modelo_bert.parameters(), 1.0)
        escalador.step(optimizador)
        escalador.update()
        programador.step()

    probs_dev = predecir_bert(datos_dev)
    f1 = f1_score(dev.y, probs_dev.argmax(1), average='macro')
    print(f'Época {epoca + 1}: macro-F1 validación = {f1:.4f} | {(time.time() - inicio) / 60:.1f} min')
    if f1 > mejor_f1:
        mejor_f1 = f1
        modelo_bert.save_pretrained(ruta_bert)
        tokenizador.save_pretrained(ruta_bert)
        print('  → Mejor hasta ahora, guardado en Drive')

print(f'\nTiempo total: {(time.time() - inicio) / 60:.1f} minutos')

Época 1/2:   0%|          | 0/2814 [00:00<?, ?it/s]

Época 1: macro-F1 validación = 0.8326 | 17.3 min


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  → Mejor hasta ahora, guardado en Drive


Época 2/2:   0%|          | 0/2814 [00:00<?, ?it/s]

Época 2: macro-F1 validación = 0.8354 | 34.6 min


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  → Mejor hasta ahora, guardado en Drive

Tiempo total: 34.6 minutos


In [ ]:
ruta_bert = f'{CARPETA_DRIVE}/pubmedbert_texto'

from sklearn.metrics import classification_report
import pandas as pd

modelo_bert = AutoModelForSequenceClassification.from_pretrained(ruta_bert).to(dispositivo)
probs_dev = predecir_bert(datos_dev)
pred_dev = probs_dev.argmax(1)

print('Exactitud:', round(accuracy_score(dev.y, pred_dev), 3))
print('Macro-F1: ', round(f1_score(dev.y, pred_dev, average='macro'), 3))
print()
print(classification_report(dev.y, pred_dev, target_names=CLASES, digits=3))

salida = dev[['abstract_id', 'indice', 'etiqueta']].copy()
salida['prediccion'] = [CLASES[i] for i in pred_dev]
for i, c in enumerate(CLASES):
    salida[f'prob_{c}'] = probs_dev[:, i]
salida.to_csv(f'{CARPETA_DRIVE}/predicciones_dev_pubmedbert_texto.csv', index=False)
print('Predicciones guardadas en Drive')

Loading weights:   0%|          | 0/201 [00:08<?, ?it/s]

Exactitud: 0.889
Macro-F1:  0.835

              precision    recall  f1-score   support

  BACKGROUND      0.694     0.828     0.755      3449
   OBJECTIVE      0.791     0.588     0.674      2376
     METHODS      0.942     0.962     0.952      9964
     RESULTS      0.940     0.928     0.934      9841
 CONCLUSIONS      0.876     0.848     0.862      4582

    accuracy                          0.889     30212
   macro avg      0.848     0.831     0.835     30212
weighted avg      0.891     0.889     0.888     30212

Predicciones guardadas en Drive
